# 持久化
> 主要支持的是短期记忆

可以实现功能：
* 对话持久化
* 断点续传
* 时间旅行调试：回溯
* 状态分支：从任意状态开始执行（创建新的执行路径）

## 核心概念

线程 和 存档点
* 线程代表图的独特执行上下文。每个线程都由唯一的 thread_id 标识。
* 存档点是图在特定时间点的状态快照。当启用持久化时，LangGraph 会在图执行的每个超步后自动创建这些快照。

## 存档点的分类
1. MemorySaver：内存存档点--开发阶段
2. SqliteSaver：SQLite 存档点--本地部署、小规模应用
3. PostgresSaver：PostgreSQL 存档点--生产环境推荐

## demo--MemorySaver

In [1]:
from langgraph.checkpoint.memory import MemorySaver
from typing import TypedDict
from langgraph.graph import MessagesState, StateGraph, END, START
from langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv

load_dotenv()

llm = ChatOpenAI(
        model=os.getenv("MODEL_NAME"),
        base_url=os.getenv("BASE_URL"),
        api_key=os.getenv("OPENAI_API_KEY"),
        temperature=0,
    )

class State(TypedDict):
    topic: str
    joke: str

def refine_topic(state: State):
    return {"topic": state["topic"] + " 和 猫咪"}

def generate_joke(state: State):
    return {"joke": f"这是一个关于 {state['topic']} 的笑话"}

def generate_joke_with_llm(state: State):
    llm_response = llm.invoke(
        [{"role": "user", "content": f"生成一个关于： {state['topic']} 的笑话"}]
    )
    return {"joke": llm_response.content}


# 使用 LLM 版本的笑话生成器，但添加持久化
graph_persistent = (
    StateGraph(State)
    .add_node(refine_topic)
    .add_node("generate_joke", generate_joke_with_llm)
    .add_edge(START, "refine_topic")
    .add_edge("refine_topic", "generate_joke")
    .compile(checkpointer=MemorySaver())  # 启用持久化
)

# 定义线程配置--在每次运行时都使用相同的线程 ID
# 同一对话或者工作流中应使用相同的线程 ID
config = {"configurable": {"thread_id": "my_thread_1"}}

print("第一次运行：")
for chunk in graph_persistent.stream(
    {"topic": "ice cream"},
    config=config,
    stream_mode="updates",
):
    print(chunk)

print("\n获取最终状态：")
print(graph_persistent.get_state(config).values)

第一次运行：
{'refine_topic': {'topic': 'ice cream 和 猫咪'}}
{'generate_joke': {'joke': '当然！这里有一个关于冰淇淋和猫咪的可爱笑话：\n\n---\n\n有一天，一只猫咪跑到冰淇淋店，对老板说：\n\n“老板，老板，来一个香草冰淇淋，不要蛋卷，要纸杯装。”\n\n冰淇淋老板很奇怪，问：“喵喵喵？你这是要做什么？”\n\n猫咪一边舔爪子一边说：\n\n“我要回去哄我家那只狗，它又生气了。”\n\n---\n\n希望你笑了！😄🍦🐱'}}

获取最终状态：
{'topic': 'ice cream 和 猫咪', 'joke': '当然！这里有一个关于冰淇淋和猫咪的可爱笑话：\n\n---\n\n有一天，一只猫咪跑到冰淇淋店，对老板说：\n\n“老板，老板，来一个香草冰淇淋，不要蛋卷，要纸杯装。”\n\n冰淇淋老板很奇怪，问：“喵喵喵？你这是要做什么？”\n\n猫咪一边舔爪子一边说：\n\n“我要回去哄我家那只狗，它又生气了。”\n\n---\n\n希望你笑了！😄🍦🐱'}
